# 01 数据探索分析

## 目标
- 加载三个数据表
- 分析数据基本情况
- 生成3张关键图表：
  1. 类别分布柱状图/饼图
  2. 关键费用分布直方图+核密度曲线
  3. 月度/周度就诊频次双折线图

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# 设置路径
ROOT = Path('.').resolve().parent
DATA_DIR = ROOT / '原始数据'
OUTPUT_DIR = ROOT / 'outputs' / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'项目根目录: {ROOT}')
print(f'数据目录: {DATA_DIR}')
print(f'输出目录: {OUTPUT_DIR}')

## 1. 数据加载

In [ ]:
# 加载三个数据表
print('加载数据中...')

# 标签数据
df_id = pd.read_csv(DATA_DIR / 'df_id_train.csv')
print(f'df_id_train: {df_id.shape}')

# 就诊记录
df_train = pd.read_csv(DATA_DIR / 'df_train.csv')
print(f'df_train: {df_train.shape}')

# 费用明细
fee_detail = pd.read_csv(DATA_DIR / 'fee_detail.csv')
print(f'fee_detail: {fee_detail.shape}')

print('\n数据加载完成!')

In [ ]:
# 查看标签数据
print('=== 标签数据 (df_id_train) ===')
print(df_id.head())
print(f'\n列名: {df_id.columns.tolist()}')

In [ ]:
# 查看就诊记录
print('=== 就诊记录 (df_train) ===')
print(f'列数: {len(df_train.columns)}')
print(f'列名:\n{df_train.columns.tolist()}')

In [ ]:
# 查看费用明细
print('=== 费用明细 (fee_detail) ===')
print(fee_detail.head())
print(f'\n列名: {fee_detail.columns.tolist()}')

## 2. 数据基本统计

In [ ]:
# 标签分布统计
label_col = df_id.columns[1]  # 第二列是标签
label_counts = df_id[label_col].value_counts()
print('=== 标签分布 ===')
print(label_counts)
print(f'\n欺诈率: {label_counts[1] / len(df_id) * 100:.2f}%')

In [ ]:
# 每个用户的就诊次数统计
visit_counts = df_train.groupby('个人编码').size()
print('=== 每人就诊次数统计 ===')
print(visit_counts.describe())

## 3. 图表1: 类别分布柱状图/饼图

In [ ]:
# 图表1: 类别分布
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 柱状图
colors = ['#2ecc71', '#e74c3c']
labels = ['正常 (0)', '欺诈 (1)']
counts = [label_counts[0], label_counts[1]]

bars = axes[0].bar(labels, counts, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_title('类别分布柱状图', fontsize=14, fontweight='bold')
axes[0].set_ylabel('样本数量', fontsize=12)
axes[0].set_xlabel('类别', fontsize=12)

# 添加数值标签
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200, 
                 f'{count:,}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# 饼图
explode = (0, 0.1)  # 突出欺诈类
wedges, texts, autotexts = axes[1].pie(counts, labels=labels, colors=colors, 
                                        explode=explode, autopct='%1.1f%%',
                                        startangle=90, shadow=True,
                                        textprops={'fontsize': 12})
axes[1].set_title('类别分布饼图', fontsize=14, fontweight='bold')

# 加粗百分比
for autotext in autotexts:
    autotext.set_fontweight('bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_类别分布图.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n图表已保存: {OUTPUT_DIR / "01_类别分布图.png"}')

## 4. 图表2: 关键费用分布直方图+核密度曲线

In [ ]:
# 转换费用列为数值类型
fee_cols = ['药品费发生金额', '检查费发生金额', '治疗费发生金额']

for col in fee_cols:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce')

# 合并标签
df_merged = df_train.merge(df_id, on='个人编码', how='left')
label_col = df_merged.columns[-1]  # 最后一列是标签

print('费用列统计:')
print(df_merged[fee_cols].describe())

In [ ]:
# 图表2: 关键费用分布
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

fee_names = ['药品费', '检查费', '治疗费']

for ax, col, name in zip(axes, fee_cols, fee_names):
    # 过滤异常值 (取99%分位数以内)
    data = df_merged[col].dropna()
    upper = data.quantile(0.99)
    data_filtered = data[data <= upper]
    
    # 分类别绘制
    normal_data = df_merged[df_merged[label_col] == 0][col].dropna()
    fraud_data = df_merged[df_merged[label_col] == 1][col].dropna()
    
    normal_data = normal_data[normal_data <= upper]
    fraud_data = fraud_data[fraud_data <= upper]
    
    # 直方图 + 核密度曲线
    ax.hist(normal_data, bins=50, alpha=0.5, label='正常', color='#2ecc71', density=True)
    ax.hist(fraud_data, bins=50, alpha=0.5, label='欺诈', color='#e74c3c', density=True)
    
    # 核密度曲线
    if len(normal_data) > 0:
        normal_data.plot.kde(ax=ax, color='#27ae60', linewidth=2, label='正常KDE')
    if len(fraud_data) > 0:
        fraud_data.plot.kde(ax=ax, color='#c0392b', linewidth=2, label='欺诈KDE')
    
    ax.set_title(f'{name}发生金额分布', fontsize=13, fontweight='bold')
    ax.set_xlabel('金额 (元)', fontsize=11)
    ax.set_ylabel('密度', fontsize=11)
    ax.legend(loc='upper right', fontsize=9)
    
    # 添加统计信息
    mean_val = data_filtered.mean()
    median_val = data_filtered.median()
    ax.axvline(mean_val, color='blue', linestyle='--', alpha=0.7, label=f'均值:{mean_val:.0f}')
    ax.axvline(median_val, color='orange', linestyle='--', alpha=0.7, label=f'中位数:{median_val:.0f}')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_费用分布直方图.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n图表已保存: {OUTPUT_DIR / "02_费用分布直方图.png"}')

## 5. 图表3: 月度/周度就诊频次双折线图

In [ ]:
# 解析时间字段
time_col = '交易时间'
df_merged[time_col] = pd.to_datetime(df_merged[time_col], errors='coerce')

# 提取月份和周次
df_merged['月份'] = df_merged[time_col].dt.to_period('M').astype(str)
df_merged['周次'] = df_merged[time_col].dt.isocalendar().week
df_merged['年份'] = df_merged[time_col].dt.year

print('时间范围:')
print(f'最早: {df_merged[time_col].min()}')
print(f'最晚: {df_merged[time_col].max()}')

In [ ]:
# 图表3: 月度/周度就诊频次
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 月度就诊频次
monthly_normal = df_merged[df_merged[label_col] == 0].groupby('月份').size()
monthly_fraud = df_merged[df_merged[label_col] == 1].groupby('月份').size()

# 对齐索引
all_months = sorted(set(monthly_normal.index) | set(monthly_fraud.index))
monthly_normal = monthly_normal.reindex(all_months, fill_value=0)
monthly_fraud = monthly_fraud.reindex(all_months, fill_value=0)

axes[0].plot(range(len(all_months)), monthly_normal.values, 'o-', 
             color='#2ecc71', linewidth=2, markersize=6, label='正常')
axes[0].plot(range(len(all_months)), monthly_fraud.values, 's-', 
             color='#e74c3c', linewidth=2, markersize=6, label='欺诈')
axes[0].set_title('月度就诊频次分布', fontsize=14, fontweight='bold')
axes[0].set_xlabel('月份', fontsize=12)
axes[0].set_ylabel('就诊次数', fontsize=12)
axes[0].set_xticks(range(len(all_months)))
axes[0].set_xticklabels(all_months, rotation=45, ha='right', fontsize=9)
axes[0].legend(loc='upper right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# 周度就诊频次 (按周次聚合)
weekly_normal = df_merged[df_merged[label_col] == 0].groupby('周次').size()
weekly_fraud = df_merged[df_merged[label_col] == 1].groupby('周次').size()

# 对齐索引 (1-52周)
all_weeks = range(1, 53)
weekly_normal = weekly_normal.reindex(all_weeks, fill_value=0)
weekly_fraud = weekly_fraud.reindex(all_weeks, fill_value=0)

axes[1].plot(all_weeks, weekly_normal.values, '-', 
             color='#2ecc71', linewidth=2, label='正常', alpha=0.8)
axes[1].plot(all_weeks, weekly_fraud.values, '-', 
             color='#e74c3c', linewidth=2, label='欺诈', alpha=0.8)
axes[1].fill_between(all_weeks, weekly_normal.values, alpha=0.2, color='#2ecc71')
axes[1].fill_between(all_weeks, weekly_fraud.values, alpha=0.2, color='#e74c3c')
axes[1].set_title('周度就诊频次分布', fontsize=14, fontweight='bold')
axes[1].set_xlabel('周次 (1-52)', fontsize=12)
axes[1].set_ylabel('就诊次数', fontsize=12)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(1, 52)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_就诊频次分布图.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n图表已保存: {OUTPUT_DIR / "03_就诊频次分布图.png"}')

## 6. 数据质量报告

In [ ]:
# 缺失值统计
print('=== df_train 缺失值统计 (Top 20) ===')
missing = df_train.isnull().sum()
missing_pct = (missing / len(df_train) * 100).round(2)
missing_df = pd.DataFrame({'缺失数': missing, '缺失率%': missing_pct})
missing_df = missing_df[missing_df['缺失数'] > 0].sort_values('缺失率%', ascending=False)
print(missing_df.head(20))

In [ ]:
# 总结
print('\n' + '='*50)
print('数据探索总结')
print('='*50)
print(f'参保人数: {len(df_id):,}')
print(f'就诊记录数: {len(df_train):,}')
print(f'费用明细数: {len(fee_detail):,}')
print(f'正常样本: {label_counts[0]:,} ({label_counts[0]/len(df_id)*100:.1f}%)')
print(f'欺诈样本: {label_counts[1]:,} ({label_counts[1]/len(df_id)*100:.1f}%)')
print(f'平均每人就诊次数: {len(df_train)/len(df_id):.1f}')
print('\n生成图表:')
print('  1. 01_类别分布图.png')
print('  2. 02_费用分布直方图.png')
print('  3. 03_就诊频次分布图.png')